# Automated Trading System — Data Analysis & ML Development

This notebook documents the development process of our ETL pipeline and Machine Learning model.
We use this notebook for exploration, then the final code lives in `src/etl.py` and `src/model.py`.

**Companies:** AAPL, MSFT, GOOG, AMZN, TSLA

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Data Extraction (ETL — Extract)

We load the raw CSV files downloaded from SimFin's bulk download.
SimFin uses semicolons (`;`) as delimiters.

In [ ]:
# Load raw data from SimFin bulk download
prices_df = pd.read_csv('../data/us-shareprices-daily.csv', sep=';')
companies_df = pd.read_csv('../data/us-companies.csv', sep=';')

print(f'Share prices: {prices_df.shape[0]} rows, {prices_df.shape[1]} columns')
print(f'Companies: {companies_df.shape[0]} rows')
print(f'\nShare prices columns: {list(prices_df.columns)}')

In [ ]:
# Preview the raw data
prices_df.head()

In [ ]:
# Check which tickers are available
TICKERS = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'TSLA']
for t in TICKERS:
    count = len(prices_df[prices_df['Ticker'] == t])
    print(f'{t}: {count} rows')

## 3. Data Transformation (ETL — Transform)

We'll develop the pipeline for **one company (AAPL)** first, then apply it to all.

In [ ]:
# Filter for AAPL
ticker = 'AAPL'
df = prices_df[prices_df['Ticker'] == ticker].copy()
print(f'Filtered {len(df)} rows for {ticker}')

# Get company info
company_info = companies_df[companies_df['Ticker'] == ticker]
print(f'Company: {company_info.iloc[0]["Company Name"]}')

### 3.1 Cleaning

In [ ]:
# Parse dates and sort
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

# Drop rows with missing Close price
df = df.dropna(subset=['Close'])

# Fill missing Volume with 0 (SimFin free tier has missing Volume data)
if 'Volume' in df.columns:
    df['Volume'] = df['Volume'].fillna(0)

# Fill missing High/Low with Close
if 'High' in df.columns:
    df['High'] = df['High'].fillna(df['Close'])
if 'Low' in df.columns:
    df['Low'] = df['Low'].fillna(df['Close'])

# Remove duplicate dates
df = df.drop_duplicates(subset=['Date'], keep='last')
df = df.reset_index(drop=True)

print(f'\nCleaned data: {len(df)} rows')
df.head()

### 3.2 Feature Engineering

We create **technical indicators** that the ML model will use as input features:
- **Returns**: Daily percentage change in price
- **SMA_5, SMA_20**: Simple Moving Averages (short/medium term trends)
- **EMA_12**: Exponential Moving Average (weights recent prices more)
- **RSI_14**: Relative Strength Index (momentum — overbought/oversold)
- **Volatility_20**: Rolling standard deviation of returns
- **Volume_Change**: Daily change in trading volume
- **High_Low_Range**: Intraday price range

In [ ]:
# Daily Returns
df['Returns'] = df['Close'].pct_change()

# Simple Moving Averages
df['SMA_5'] = df['Close'].rolling(window=5).mean()
df['SMA_20'] = df['Close'].rolling(window=20).mean()

# Exponential Moving Average
df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()

# RSI (Relative Strength Index)
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0.0)
loss = (-delta).where(delta < 0, 0.0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
rs = avg_gain / avg_loss
df['RSI_14'] = 100 - (100 / (1 + rs))

# Volatility
df['Volatility_20'] = df['Returns'].rolling(window=20).std()

# Volume Change
if 'Volume' in df.columns and df['Volume'].sum() > 0:
    df['Volume_Change'] = df['Volume'].pct_change().replace([np.inf, -np.inf], 0).fillna(0)
else:
    df['Volume_Change'] = 0.0

# High-Low Range
df['High_Low_Range'] = df['High'] - df['Low']

print('Features created!')
df[['Date', 'Close', 'Returns', 'SMA_5', 'SMA_20', 'RSI_14', 'Volatility_20']].tail(10)

### 3.3 Visualize the Data

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot 1: Close price with moving averages
axes[0].plot(df['Date'], df['Close'], label='Close', linewidth=1)
axes[0].plot(df['Date'], df['SMA_5'], label='SMA 5', linewidth=0.8, alpha=0.7)
axes[0].plot(df['Date'], df['SMA_20'], label='SMA 20', linewidth=0.8, alpha=0.7)
axes[0].set_title(f'{ticker} — Close Price & Moving Averages')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: RSI
axes[1].plot(df['Date'], df['RSI_14'], color='purple', linewidth=0.8)
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
axes[1].set_title('RSI (14)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Volatility
axes[2].plot(df['Date'], df['Volatility_20'], color='orange', linewidth=0.8)
axes[2].set_title('Volatility (20-day)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.4 Create Target Variable

Our target is **binary classification**:
- `1` = tomorrow's Close price is **higher** than today's (UP)
- `0` = tomorrow's Close price is **lower or equal** to today's (DOWN)

In [ ]:
# Create target: 1 if next day's Close > today's Close
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

print('Target distribution:')
print(df['Target'].value_counts())
print(f'\nUp days: {df["Target"].sum()} ({df["Target"].mean()*100:.1f}%)')
print(f'Down days: {(df["Target"] == 0).sum()} ({(1-df["Target"].mean())*100:.1f}%)')

### 3.5 Drop Missing Rows & Save

In [ ]:
# Define feature columns
FEATURE_COLUMNS = [
    'Returns', 'SMA_5', 'SMA_20', 'EMA_12',
    'RSI_14', 'Volatility_20', 'Volume_Change', 'High_Low_Range'
]

# Drop rows with NaN in essential columns
essential = FEATURE_COLUMNS + ['Target']
rows_before = len(df)
df = df.dropna(subset=essential).reset_index(drop=True)
rows_after = len(df)

print(f'Dropped {rows_before - rows_after} rows with NaN values')
print(f'Final dataset: {rows_after} rows')
df.head()

---

## 4. Machine Learning Model

We use **Logistic Regression** for binary classification (UP/DOWN).

**Why Logistic Regression?**
- Simple and interpretable
- Works well as a baseline for binary classification
- Fast to train
- Outputs probabilities (confidence)

### 4.1 Prepare Data

In [ ]:
# Separate features (X) and target (y)
X = df[FEATURE_COLUMNS].copy()
y = df['Target'].copy()

# Replace infinities
X = X.replace([np.inf, -np.inf], np.nan)
mask = X.notna().all(axis=1)
X = X[mask]
y = y[mask]

# Split into train (80%) and test (20%)
# shuffle=False because this is time-series data — we train on past, test on future
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f'Training set: {len(X_train)} rows')
print(f'Test set: {len(X_test)} rows')
print(f'Target balance (train): {dict(y_train.value_counts())}')

In [ ]:
# Scale features — important for Logistic Regression
# StandardScaler transforms features to have mean=0 and std=1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Learn scaling from train data
X_test_scaled = scaler.transform(X_test)          # Apply same scaling to test data

print('Features scaled!')
print(f'Example (before): {X_train.iloc[0].values[:3]}')
print(f'Example (after):  {X_train_scaled[0][:3]}')

### 4.2 Train the Model

In [ ]:
# Train Logistic Regression
model = LogisticRegression(
    max_iter=1000,             # Max iterations for solver to converge
    random_state=42,           # Seed for reproducibility
    class_weight='balanced'    # Handle class imbalance
)
model.fit(X_train_scaled, y_train)

print('Model trained!')

### 4.3 Evaluate the Model

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test_scaled)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)')

# Classification Report
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['DOWN (0)', 'UP (1)']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(f'                 Predicted DOWN  Predicted UP')
print(f'  Actual DOWN    {cm[0][0]:>13}  {cm[0][1]:>12}')
print(f'  Actual UP      {cm[1][0]:>13}  {cm[1][1]:>12}')

In [ ]:
# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['DOWN', 'UP'])
ax.set_yticklabels(['DOWN', 'UP'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'{ticker} — Confusion Matrix')

# Add text annotations
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i][j]), ha='center', va='center', fontsize=16, fontweight='bold')

plt.colorbar(im)
plt.tight_layout()
plt.show()

### 4.4 Feature Importance

Which features contribute most to the model's predictions?

In [ ]:
# Logistic Regression coefficients = feature importance
importance = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if c > 0 else 'red' for c in importance['Coefficient']]
ax.barh(importance['Feature'], importance['Coefficient'], color=colors)
ax.set_xlabel('Coefficient (positive = predicts UP, negative = predicts DOWN)')
ax.set_title(f'{ticker} — Feature Importance (Logistic Regression Coefficients)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.5 Export Model

We save the model and scaler as `.pkl` files so the web app can load them.

In [ ]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(model, f'../models/{ticker}_model.pkl')
joblib.dump(scaler, f'../models/{ticker}_scaler.pkl')

print(f'Model saved to models/{ticker}_model.pkl')
print(f'Scaler saved to models/{ticker}_scaler.pkl')

---

## 5. Summary

| Step | Description |
|---|---|
| **Extract** | Loaded 6M+ rows of share price data from SimFin bulk CSV |
| **Transform** | Cleaned data, filled missing values, created 8 technical indicators |
| **Target** | Binary classification: UP (1) or DOWN (0) based on next day's close |
| **Model** | Logistic Regression with StandardScaler, 80/20 time-series split |
| **Export** | Model + scaler saved as .pkl files for the Streamlit web app |

**Note:** Stock prediction accuracy around 50% is expected — the goal of this project is the Python engineering, not prediction quality.

The final scripts (`src/etl.py` and `src/model.py`) automate this entire process for all 5 tickers.